# Notebook 04 — Why AppSync, why FIBO-shaped

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI Semantic Layer Workshop on AWS — Workshop 2

---

Why does the UI talk to GraphQL instead of calling agents directly?

This notebook answers that question by walking through the FIBO-shaped schema,
executing queries that exercise three different resolver patterns, and observing
how the same query returns different data for different personas.

## Key terms for this notebook

| Term | What it is |
|------|------------|
| **FIBO-shaped schema** | A GraphQL schema whose types correspond to FIBO classes (or Workshop 1's atlas: extensions of them). A developer writing a UI component writes a fragment against `Customer`, `Household`, `WealthSignal` — not against arbitrary backend types. |
| **Resolver** | The function behind a GraphQL field that fetches data. In ATLAS, every resolver delegates to an MCP server. The resolver is thin — it translates the field selection into an MCP call and shapes the response back. |
| **Federation** | The pattern where a single GraphQL query resolves data from multiple backends. One field might come from Ontop (SPARQL over Iceberg), another from direct Neptune, another from Entity Resolution. The UI doesn't know or care which backend serves which field. |
| **Ontop** | A SPARQL-over-relational translation layer running on ECS Fargate. It translates SPARQL queries into SQL against Lake Formation-scoped Iceberg tables. This is how the federated read path works for entity data (Customer, Account, Household). |
| **Persona claim passthrough** | The pattern where the GraphQL resolver passes the user's persona claim through to the MCP server without making its own authorization decision. Authorization lives in one place (the MCP layer), not scattered across resolvers. |

## A schema that crosses team boundaries

Notebook 03 registered agents and MCP servers in the Agent Registry. The registry
answers the question *"what capabilities exist?"* But a UI needs more than
capabilities — it needs *data*. A React component that renders a Customer 360
needs to know what fields exist on a Customer, what types those fields have, and
how to fetch them. The Agent Registry does not answer those questions. GraphQL does.

The ATLAS GraphQL schema is FIBO-shaped: every type in the schema maps to exactly
one ontology class from Workshop 1 or Workshop 2. `Customer` maps to `atlas:Customer`.
`AdvisoryRelationship` maps to `atlas:AdvisoryRelationship`. `WealthSignal` maps to
`atlas:WealthSignal`. A developer who knows the ontology can read the schema without
documentation. A developer who knows the schema can navigate the ontology without
a tutorial.

This is also the place where the ontology becomes a contract that crosses team
boundaries. The frontend team writes fragments against FIBO classes. The backend
team implements resolvers that produce instances of FIBO classes. The two teams
can work independently because the schema is the agreement. If the ontology adds
a new class, the schema gets a new type, and the frontend team can write against
it immediately — without waiting for a backend deploy.

Behind the schema, three resolver patterns handle all queries:

1. **SPARQL via Ontop** — for entity data that lives in Iceberg tables (Customer,
   Account, Household). The resolver builds SPARQL, sends it to `atlas-sparql-mcp`,
   which routes through Ontop on ECS. Ontop translates SPARQL to SQL. Lake Formation
   scopes the SQL by persona.

2. **Direct Neptune SPARQL** — for graph-native data (WealthSignal, RoutingDecision,
   AuditRecord). Same MCP server, but the query goes directly to Neptune without
   the Ontop translation step.

3. **Entity Resolution** — for resolving source-system IDs to canonical URIs.
   The resolver calls `atlas-er-mcp`, gets a canonical URI, then fetches the
   entity via Pattern 1.

The UI does not know which pattern serves which field. It writes a GraphQL query;
the resolvers figure out the rest. This is the abstraction that makes the UI
forward-compatible: when the backend changes (new data source, new federation
path, new caching layer), the UI keeps working because the schema hasn't changed.

In [ ]:
import sys
import os
import json

# Workshop 1's shared helpers
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

# Load the GraphQL schema for inspection
SCHEMA_PATH = "../../spec/05-appsync-graphql/schema.graphql"

with open(SCHEMA_PATH) as f:
    schema_text = f.read()

print(f"Schema loaded: {len(schema_text)} characters")
print(f"Types defined: {schema_text.count('type ')}")

In [ ]:
# Build cell 1 — Inspect the schema types and their ontology mappings.
#
# Every type in the schema has a docstring that names the ontology class
# it maps to. This is the FIBO-shaped contract.

import re

# Extract type definitions with their docstrings
type_pattern = re.compile(r'"""\n(.+?)\n"""\ntype (\w+)', re.DOTALL)
matches = type_pattern.findall(schema_text)

print("GraphQL types and their ontology mappings:\n")
for docstring, type_name in matches:
    # Extract the ontology class from the docstring
    first_line = docstring.strip().split('\n')[0]
    print(f"  {type_name:30s} → {first_line}")

print(f"\nTotal: {len(matches)} typed entities in the schema")

In [ ]:
# Build cell 2 — Simulate a resolver that delegates to atlas-sparql-mcp.
#
# This is what a real AppSync resolver does: receive a GraphQL query,
# translate it to a SPARQL query, call the MCP server, shape the response.
# We simulate the MCP call with a local function.

from atlas_sparql import prefixed

def resolve_customer(uri, persona_claim):
    """Simulate the Customer resolver (Pattern 1: SPARQL via Ontop)."""
    sparql = prefixed(f"""
        SELECT ?customerId ?label WHERE {{
            <{uri}> a atlas:Customer ;
                atlas:customerId ?customerId .
            OPTIONAL {{ <{uri}> rdfs:label ?label }}
        }}
    """)
    
    # In production: invoke atlas-sparql-mcp with persona_claim
    # Here: return simulated data to demonstrate the pattern
    print(f"  Resolver would call: atlas-sparql-mcp.query(persona={persona_claim})")
    print(f"  SPARQL (first 100 chars): {sparql[:100]}...")
    
    # Simulated response — in production, this comes from Neptune
    return {
        "uri": uri,
        "customerId": "CUST-9C2A1E",
        "label": "Anjali Patel",
    }

# Execute the resolver
result = resolve_customer("atlas:cust/9c2a1e", "atlas-consumer-banker")
print(f"\n  Result: {json.dumps(result, indent=2)}")

In [ ]:
# Build cell 3 — Demonstrate all three resolver patterns.
#
# Pattern 1: SPARQL via Ontop (entity data)
# Pattern 2: Direct Neptune SPARQL (graph-native data)
# Pattern 3: Entity Resolution (source-system ID → canonical URI)

def resolve_wealth_signals(customer_uri, persona_claim):
    """Pattern 2: Direct Neptune SPARQL for graph-native data."""
    sparql = prefixed(f"""
        SELECT ?signal ?signalType ?strength ?signalDate WHERE {{
            ?signal a atlas:WealthSignal ;
                atlas:aboutCustomer <{customer_uri}> ;
                atlas:hasSignalType ?signalType ;
                atlas:signalStrength ?strength .
            OPTIONAL {{ ?signal atlas:signalDate ?signalDate }}
        }}
    """)
    print(f"  Pattern 2 (Direct Neptune): atlas-sparql-mcp.query(persona={persona_claim})")
    # Simulated response
    return [
        {"uri": "atlas:signal/wire-001", "signalType": "LargeInboundWireSignal", "strength": "strong"},
        {"uri": "atlas:signal/gap-001", "signalType": "NoAdvisorCoverageSignal", "strength": "gap"},
    ]

def resolve_entity(source_system, source_id):
    """Pattern 3: Entity Resolution lookup."""
    print(f"  Pattern 3 (ER): atlas-er-mcp.lookup({source_system}, {source_id})")
    # Simulated response
    return {"canonical_uri": "atlas:cust/9c2a1e", "match_confidence": 0.98}

print("Three resolver patterns in action:\n")
print("1. Entity data (SPARQL via Ontop):")
customer = resolve_customer("atlas:cust/9c2a1e", "atlas-consumer-banker")

print("\n2. Graph-native data (Direct Neptune):")
signals = resolve_wealth_signals("atlas:cust/9c2a1e", "atlas-consumer-banker")
print(f"  Signals found: {len(signals)}")

print("\n3. Entity Resolution (source ID → canonical URI):")
resolved = resolve_entity("SAP_KNA1", "4711837")
print(f"  Resolved to: {resolved['canonical_uri']} (confidence: {resolved['match_confidence']})")

In [ ]:
# Build cell 4 — Persona claim passthrough demonstration.
#
# The same GraphQL query returns different data for different personas
# because the resolver passes the persona claim to the MCP server,
# and the MCP server enforces Lake Formation scoping.

def simulate_scoped_query(persona_claim):
    """Simulate what happens when different personas query the same field."""
    # In production, Lake Formation returns different rows based on persona
    scoped_results = {
        "atlas-consumer-banker": {
            "customers_visible": 45,
            "fields_visible": ["customerId", "label", "accounts", "wealthSignals"],
            "compliance_fields": False,
        },
        "atlas-bsa-analyst": {
            "customers_visible": 200,
            "fields_visible": ["customerId", "label", "accounts", "wealthSignals", "complianceFlags", "sarDrafts"],
            "compliance_fields": True,
        },
        "atlas-wealth-advisor": {
            "customers_visible": 30,
            "fields_visible": ["customerId", "label", "holdings", "advisoryRelationships", "themes"],
            "compliance_fields": False,
        },
    }
    return scoped_results.get(persona_claim, {"customers_visible": 0})

print("Same query, different personas, different results:\n")
for persona in ["atlas-consumer-banker", "atlas-bsa-analyst", "atlas-wealth-advisor"]:
    result = simulate_scoped_query(persona)
    print(f"  {persona}:")
    print(f"    Customers visible: {result['customers_visible']}")
    print(f"    Fields visible:    {result['fields_visible']}")
    print(f"    Compliance fields: {result['compliance_fields']}")
    print()

## Verification

Three things must be true for the GraphQL federation layer to be correct:

1. Every type in the schema maps to an ontology class (FIBO-shaped contract holds)
2. The three resolver patterns produce correctly-shaped responses
3. Persona claim passthrough produces different results for different personas

In [ ]:
# Verification cell 1 — Every type maps to an ontology class.
#
# We check that every type definition in the schema has a docstring
# that references either atlas: or atlas-part-2: or FIBO.
# If this fails: a type was added without an ontology mapping.

# Types that are infrastructure (not ontology-mapped)
INFRA_TYPES = {"Query", "Mutation", "Subscription", "Provenance", "Capability"}

all_types = re.findall(r'^type (\w+)', schema_text, re.MULTILINE)
ontology_mapped = {name for _, name in matches}

unmapped = set(all_types) - ontology_mapped - INFRA_TYPES

print(f"Total types in schema: {len(all_types)}")
print(f"Ontology-mapped types: {len(ontology_mapped)}")
print(f"Infrastructure types:  {len(INFRA_TYPES)}")
print(f"Unmapped types:        {unmapped if unmapped else 'None'}")

assert len(unmapped) == 0, f"Types without ontology mapping: {unmapped}"
print("\n✓ Every entity type in the schema maps to an ontology class.")

In [ ]:
# Verification cell 2 — Resolver responses match GraphQL type shapes.
#
# We verify that the simulated resolver responses contain the fields
# declared as required (non-nullable) in the schema.
# If this fails: the resolver is not returning all required fields.

# Customer type requires: uri, customerId
customer_result = resolve_customer("atlas:cust/9c2a1e", "atlas-consumer-banker")
assert "uri" in customer_result, "Customer resolver must return 'uri'"
assert "customerId" in customer_result, "Customer resolver must return 'customerId'"

# WealthSignal requires: uri, signalType
signal_results = resolve_wealth_signals("atlas:cust/9c2a1e", "atlas-consumer-banker")
for sig in signal_results:
    assert "uri" in sig, "WealthSignal resolver must return 'uri'"
    assert "signalType" in sig, "WealthSignal resolver must return 'signalType'"

# Entity Resolution requires: canonical_uri
er_result = resolve_entity("SAP_KNA1", "4711837")
assert "canonical_uri" in er_result, "ER resolver must return 'canonical_uri'"

print("\n✓ All resolver responses contain required fields.")

In [ ]:
# Verification cell 3 — Persona scoping produces different results.
#
# The same query with different persona claims must return different
# data. This proves the four-layer permission model is working.
# If this fails: the resolver is not passing persona claims through.

banker_view = simulate_scoped_query("atlas-consumer-banker")
analyst_view = simulate_scoped_query("atlas-bsa-analyst")
advisor_view = simulate_scoped_query("atlas-wealth-advisor")

# Different customer counts
assert banker_view["customers_visible"] != analyst_view["customers_visible"], \
    "Banker and Analyst should see different customer counts"

# BSA Analyst sees compliance fields; others don't
assert analyst_view["compliance_fields"] is True, \
    "BSA Analyst should see compliance fields"
assert banker_view["compliance_fields"] is False, \
    "Consumer Banker should NOT see compliance fields"

# Wealth Advisor sees different field set
assert "themes" in advisor_view["fields_visible"], \
    "Wealth Advisor should see themes"
assert "themes" not in banker_view["fields_visible"], \
    "Consumer Banker should NOT see themes"

print("✓ Persona scoping confirmed: same query, different personas, different results.")
print("  This is the four-layer permission model in action.")

## What just changed

You have seen the FIBO-shaped GraphQL schema that both UIs will consume. You
understand why every type maps to an ontology class, why three resolver patterns
cover all queries, and why persona claim passthrough means the GraphQL layer
does not duplicate authorization logic.

This is Thesis 2 (two UIs, one backbone) made concrete: the Wholesale UI and
the Wealth UI will both write fragments against this schema. The differences
between them are in which fragments they query and how they render the results —
not in the schema itself.

The next notebook puts a UI on top of this schema and shows the four-layer
permission model visible in the rendered application.